# Tiny Dreamer Highway — Colab Git + Drive Setup

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

This notebook follows the project rule: GitHub is the source of truth for code, and Google Drive stores artifacts such as checkpoints, plots, and videos.

## Runtime flow

1. Mount Google Drive.
2. Clone or pull the latest repository snapshot from GitHub.
3. Install the package and dependencies.
4. Run a small sanity check before training.
5. Save outputs into the Drive-backed `artifacts/` folder.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
WORKTREE = Path('/content/CSC_580_Final_Project')

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'checkpoints', ARTIFACT_ROOT / 'media', ARTIFACT_ROOT / 'logs']:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

print('Artifact root:', ARTIFACT_ROOT)

print('Repository URL:', REPO_URL)


Mounted at /content/drive
Drive root: /content/drive/MyDrive/CSC_580_Final_Project
Artifact root: /content/drive/MyDrive/CSC_580_Final_Project/artifacts
Repository URL: https://github.com/estmon8u/CSC_580_Final_Project.git


In [2]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi

Cloning into '/content/CSC_580_Final_Project'...


In [ ]:
%%bash
set -e
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip
python -m pip install --upgrade --force-reinstall "numpy>=1.26.4,<2.0" "scipy>=1.13.1,<1.15"
python -m pip install -e .


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Obtaining file:///content/CSC_580_Final_Project
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 157.7 MB/s  0:00:00
  Building editable for tiny-dreamer-highway (pyproje

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.


## Important note after installation



If Colab updates `numpy` or `scipy` during installation, use **Runtime -> Restart session** once, then rerun the notebook from the top. This avoids binary mismatch errors when importing `highway_env` and `scipy`.


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.cli import summarize_config

config_path = PROJECT_ROOT / 'examples' / 'base_experiment.yaml'
config = load_experiment_config(config_path)
print(summarize_config(config))

Loaded config for highway-v0 | obs=64x64 | replay_capacity=10000 | sequence_length=8 | batch_size=4


In [5]:
from tiny_dreamer_highway.envs.highway_factory import make_highway_env

env = make_highway_env(config.env)
observation, info = env.reset()
action = env.action_space.sample()
next_observation, reward, terminated, truncated, info = env.step(action)
env.close()

print('Observation shape:', getattr(observation, 'shape', None))
print('Action shape:', getattr(action, 'shape', None))
print('Reward:', reward)
print('Episode ended:', bool(terminated or truncated))

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

## Next step

After this sanity pass succeeds, add cells for replay warm-start collection, world model smoke tests, and training artifacts written to `ARTIFACT_ROOT`.